# Removing the backdoor by distillation

The poisoned model from notebook 03 becomes a teacher. A student is initialised
from it with part of each layer re-randomised --- little in the shallow layers,
which hold generic filters the clean task needs, and a lot in the deep layers,
where the trigger is bound to the target class --- and then trained on clean
data to match the teacher's outputs.

    W*_l = (1 - m_l) . W_teacher_l + m_l . W_random_l,     m_l ~ Bernoulli(delta_l)

Two distillation losses are compared: hard labels (the teacher's argmax) and
soft labels (KL against the teacher's temperature-softened distribution).

**The student is probed with the same `tl.Trigger` the teacher was poisoned
with**, and ASR still excludes true airplanes. Both points sound like
bookkeeping and are the entire result: the version of this experiment this
repository is built from used a different trigger in this notebook than in the
attack --- a checkerboard in the top-right rather than a solid patch in the
bottom-right --- and measured ASR over the whole test set. The 96% -> 11% drop
it reported is what you get from probing a pattern the model never saw, against
a metric with a 10% floor. The last cell here reproduces that measurement
alongside the correct one.

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.models import resnet18

import trustlab as tl

tl.set_seed(tl.SEED)
DEVICE = tl.DEVICE

TARGET_CLASS = 0
BATCH_SIZE = 128
DISTILL_EPOCHS = 30
LR = 1e-3
TEMPERATURE = 4.0

TRIGGER = tl.Trigger(size=3, corner="br", pattern="solid")   # as in notebook 03

In [ ]:
teacher = resnet18(weights=None, num_classes=10).to(DEVICE)
teacher.load_state_dict(torch.load("poisoned_resnet18.pth", map_location=DEVICE))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

# The student trains on clean data, and augmentation helps it generalise --- it
# is not poisoned, so shifting a corner is harmless here.
clean_train = tl.cifar10(train=True, augment=True)
testset = tl.cifar10(train=False)
train_loader = DataLoader(clean_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

baseline = tl.evaluate_backdoor(teacher, test_loader, TRIGGER, TARGET_CLASS)
print(f"teacher:  clean {baseline['clean_acc']:.2f}%   "
      f"ASR {baseline['asr']:.2f}%")

In [ ]:
def distil(kind):
    tl.set_seed(tl.SEED)
    student = resnet18(weights=None, num_classes=10).to(DEVICE)
    student = tl.adaptive_initialize(teacher, student, rng=tl.SEED)
    optimizer = torch.optim.Adam(student.parameters(), lr=LR)

    history = {"loss": [], "clean": [], "asr": []}
    for epoch in range(DISTILL_EPOCHS):
        student.train()
        running = 0.0
        for images, _ in train_loader:
            images = images.to(DEVICE)
            with torch.no_grad():
                teacher_logits = teacher(images)
            optimizer.zero_grad()
            loss = tl.distillation_loss(student(images), teacher_logits,
                                        kind=kind, temperature=TEMPERATURE)
            loss.backward()
            optimizer.step()
            running += loss.item() * images.size(0)

        scores = tl.evaluate_backdoor(student, test_loader, TRIGGER, TARGET_CLASS)
        history["loss"].append(running / len(clean_train))
        history["clean"].append(scores["clean_acc"])
        history["asr"].append(scores["asr"])
        print(f"[{kind}] epoch {epoch+1:2d}/{DISTILL_EPOCHS}  "
              f"loss {history['loss'][-1]:.4f}  "
              f"clean {scores['clean_acc']:.2f}%  ASR {scores['asr']:.2f}%")
    return student, history

student_ce, hist_ce = distil("ce")

In [ ]:
student_kl, hist_kl = distil("kl")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for hist, name in [(hist_ce, "cross-entropy"), (hist_kl, "KL divergence")]:
    axes[0].plot(hist["loss"], marker="o", ms=3, label=name)
    axes[1].plot(hist["clean"], marker="o", ms=3, label=name)
    axes[2].plot(hist["asr"], marker="o", ms=3, label=name)
axes[0].set_title("distillation loss")
axes[1].set_title("clean accuracy (%)")
axes[1].axhline(baseline["clean_acc"], ls="--", c="grey", label="teacher")
axes[2].set_title("attack success rate (%)")
axes[2].axhline(baseline["asr"], ls="--", c="grey", label="teacher")
axes[2].set_ylim(0, 100)
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
print(f"{'model':<26} {'clean %':>9} {'ASR %':>9}")
print(f"{'teacher (poisoned)':<26} {baseline['clean_acc']:>9.2f} "
      f"{baseline['asr']:>9.2f}")
for student, name in [(student_ce, "student, cross-entropy"),
                      (student_kl, "student, KL divergence")]:
    s = tl.evaluate_backdoor(student, test_loader, TRIGGER, TARGET_CLASS)
    print(f"{name:<26} {s['clean_acc']:>9.2f} {s['asr']:>9.2f}")

## Reproducing the measurement error

Three ways of asking the same student the same question. The first is the one
above. The second changes only the denominator. The third changes the trigger
to the checkerboard-in-the-top-right that the original probed with --- a
pattern no model in this repository has ever been trained on, so whatever it
returns is a property of an untouched network, not of a defence.

In [ ]:
WRONG_TRIGGER = tl.Trigger(size=3, corner="tr", pattern="checker")

@torch.no_grad()
def asr_whole_testset(model, trigger):
    model.eval()
    hits = total = 0
    for images, labels in test_loader:
        images = trigger.apply(images.to(DEVICE))
        hits += (model(images).argmax(1) == TARGET_CLASS).sum().item()
        total += labels.size(0)
    return 100.0 * hits / total

print(f"{'measurement':<52} {'CE':>8} {'KL':>8}")
rows = [
    ("correct trigger, excluding true airplanes",
     lambda m: tl.evaluate_backdoor(m, test_loader, TRIGGER, TARGET_CLASS)["asr"]),
    ("correct trigger, whole test set",
     lambda m: asr_whole_testset(m, TRIGGER)),
    ("wrong trigger (top-right checker), whole test set",
     lambda m: asr_whole_testset(m, WRONG_TRIGGER)),
]
for label, fn in rows:
    print(f"{label:<52} {fn(student_ce):>8.2f} {fn(student_kl):>8.2f}")

print(f"\nthe same three, on an untrained network, for reference:")
naive = resnet18(weights=None, num_classes=10).to(DEVICE)
for label, fn in rows:
    print(f"  {label:<50} {fn(naive):>8.2f}")

The third row is the number the original reported, and the reference line under
it is the point: a network that has never seen any trigger produces something
in the same neighbourhood, because the metric's floor is the target class's
share of the test set and a checkerboard in an unused corner is close to no
input at all.

None of this says distillation cannot remove a backdoor --- the first row is
the measurement that would settle that, and it is what this notebook reports.
It says that the evidence offered for it was not evidence, and that the
difference between the two came down to writing a 3x3 array out by hand twice.